# El halo que el modelo no reproduce

**Objeto:** ROXs12b  |  **Run:** `ROXs12b_realigned`  |  **Spec:** [`docs/spec_C1_codex_chromatic_psf.md`](../../../docs/spec_C1_codex_chromatic_psf.md)

Este notebook cuenta una investigación entera, con sus figuras. Empezó por un síntoma concreto —un método de extracción detectaba señal **donde no se había inyectado ninguna**— y acabó en el modelo de PSF.

**El resultado, por delante:** el modelo reproduce el **núcleo** con un error del orden del 4 % y falla el **halo** entre el 17 % y el 107 %, con un error que crece con el radio y cambia con λ. Cuatro explicaciones se descartaron con medida y la que queda no se arregla con ninguna perilla.

### El recorrido

| § | pregunta | veredicto |
|---|---|---|
| 3 | ¿para qué usa la cadena la PSF? | dos usos, y solo uno sufre |
| 4 | ¿dónde falla el modelo, y cuánto? | crece con el radio, cambia con λ |
| 5 | ¿lo crea la combinación de exposiciones? | **no**: por exposición es peor |
| 6 | ¿es el radio de normalización? | **no**: mueve 6 puntos sobre 107 |
| 7 | ¿es la ponderación del ajuste? | **no**: ya está en su mejor opción |
| 8 | ¿lo arregla el término híbrido? | a medias, y solo generaliza la mitad |
| 9 | ¿a quién le importa? | a los métodos que restan el modelo, no a los demás |

> **Cómo leerlo.** Cada sección enseña la figura y **debajo el número medido**. Ninguna afirmación de la narrativa está escrita a mano: todas se recalculan al ejecutar, así que si la cadena cambia, el texto se cae con ella.


## 1 · Preparación

Todo sale del run que se declara abajo. El notebook no escribe nada.


In [ ]:
import csv, json, sys, warnings
from pathlib import Path

import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt

import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 120
mpl.rcParams['savefig.dpi'] = 200
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('retina')
except Exception:
    pass
_aqui = Path.cwd()
ROOT = next(p for p in (_aqui, *_aqui.parents) if (p / 'musepipe').is_dir())
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / 'notebooks'))
import _nbcommon as nb

RUN_ID = nb.resolve_run_id('ROXs12b_realigned')
RD = nb.run_dir(RUN_ID); SD = RD / 'stages'
TARGET = nb.run_target(RUN_ID) or nb.display_name(RUN_ID)
print('objeto :', TARGET, '·', nb.display_name(RUN_ID))
print('run    :', RUN_ID)


## 2 · Perillas y anclaje

Las perillas salen del **config resuelto de la etapa**, nunca copiadas como literales: C1 rellena defaults que el run no escribe.

Y antes de contar nada, el anclaje. Este notebook reconstruye el modelo de PSF y mide sobre él; si esa reconstrucción no fuera la de la cadena, todo lo demás sería una historia sobre otro modelo. Se comprueba contra dos productos reales: el cociente núcleo/total bin a bin contra **`stage_e01_qc.json`**, y el perfil radial de un bin contra **`psf_hybrid_residual.fits`**.


In [ ]:
from musepipe.stages.stage_e01_psf import stage_e01_config_from_run

E01 = stage_e01_config_from_run(RUN_ID, project_root=ROOT)
R_NORM = float(E01.get('psf_norm_radius_px', 25.0))
BANDAS = [(4800, 5533), (5533, 6267), (6267, 7000),
          (7000, 7733), (7733, 8467), (8467, 9200)]
RADIOS_MUESTRA = (30.0, 40.0, 45.0, 50.0, 60.0, 71.0)
ANCLAJES_PRUEBA = (25.0, 45.0, 60.0)   # para la §6
CSV_POR_EXPOSICION = ROOT / 'reports' / 'psf_perexp' / 'residuo_radial_por_exposicion.csv'
CSV_PEDESTAL = ROOT / 'reports' / 'psf_perexp' / 'pedestal_por_exposicion.csv'

QC_C1 = json.loads((SD / 'stage_e01_qc.json').read_text(encoding='utf-8'))
PSF_ENTREGADO = json.loads((SD / 'psf_model.json').read_text(encoding='utf-8'))
ES_MEZCLA = str(PSF_ENTREGADO.get('form', '')).lower() == 'mixture'
_RUTA_ANALITICO = SD / ('psf_model_combined.json' if ES_MEZCLA else 'psf_model.json')
if ES_MEZCLA and not _RUTA_ANALITICO.exists():
    raise FileNotFoundError(
        f'C1 entregó una mezcla pero falta {_RUTA_ANALITICO.name}: sin el ajuste'
        ' analítico al combinado estas secciones no tienen qué auditar.')
PSF_MODEL = (json.loads(_RUTA_ANALITICO.read_text(encoding='utf-8'))
             if ES_MEZCLA else PSF_ENTREGADO)
if ES_MEZCLA:
    print(f'C1 entregó una MEZCLA de {PSF_ENTREGADO["n_components"]} exposiciones'
          f' (psf_scope={PSF_ENTREGADO.get("psf_scope")!r}).')
    print(f'   lo que auditan estas secciones es el ajuste analítico al combinado:'
          f' {_RUTA_ANALITICO.name}')
print(f'\nradio de normalización: {R_NORM:.0f} px')
print(f'radio de ajuste de C1 : {QC_C1["fit"]["fit_radius_px"]:.0f} px')


### 2.a · El código copiado, y su chequeo de deriva

Las funciones de abajo son **literalmente** las de `musepipe`, copiadas para poder editarlas aquí sin tocar la cadena. La celda siguiente comprueba que siguen siendo las mismas.


In [ ]:
# ------------------------------------------------------------------
# COPIA EDITABLE. Fuente: musepipe (ver el chequeo de deriva abajo).
# ------------------------------------------------------------------
from musepipe.stats import finite_percentile
from scipy.ndimage import gaussian_filter1d
import functools
import math
import numpy as np
import warnings

MOFFAT_BETA_FLOOR = 1.05  # Moffat is only a normalizable PSF for beta > 1.
_PSFAO_PARAM_NAMES = ("r0", "C", "A", "alpha", "ratio", "theta", "beta")
PSF_SHAPE_PARAMS = ("y0", "x0", "fwhm_maj", "fwhm_min", "theta_deg", "beta")
MIXTURE_FORM = "mixture"


def source_mask(shape, centers_yx, radius_px):
    yy, xx = np.indices(shape, dtype=np.float64)
    mask = np.zeros(shape, dtype=bool)
    for center in centers_yx or ():
        if center is None:
            continue
        y, x = map(float, center)
        mask |= (yy - y) ** 2 + (xx - x) ** 2 <= float(radius_px) ** 2
    return mask


def companion_ring_metric(image, model, primary_yx, companion_yx, *, width_px=3.0, source_exclusion_radius_px=0.0):
    img = np.asarray(image, dtype=np.float64)
    mod = np.asarray(model, dtype=np.float64)
    yy, xx = np.indices(img.shape, dtype=np.float64)
    py, px = map(float, primary_yx)
    cy, cx = map(float, companion_yx)
    radius = math.hypot(cy - py, cx - px)
    rr = np.sqrt((yy - py) ** 2 + (xx - px) ** 2)
    ann = np.abs(rr - radius) <= float(width_px) / 2.0
    if source_exclusion_radius_px and source_exclusion_radius_px > 0:
        ann &= (yy - cy) ** 2 + (xx - cx) ** 2 > float(source_exclusion_radius_px) ** 2
    halo = np.abs(mod)
    vals = np.abs(img - mod) / np.maximum(halo, np.nanmedian(halo[ann]) * 0.05)
    vals = vals[ann & np.isfinite(vals)]
    if vals.size == 0:
        return {"radius_px": float(radius), "median_pct": np.nan, "p90_pct": np.nan}
    return {
        "radius_px": float(radius),
        "median_pct": float(100.0 * np.nanmedian(vals)),
        "p90_pct": float(100.0 * finite_percentile(vals, 90.0)),
    }


def radial_hybrid_profile(residual, center_yx, *, mask=None, bin_width_px=1.0, smoothing_scale_px=4.0):
    resid = np.asarray(residual, dtype=np.float64)
    yy, xx = np.indices(resid.shape, dtype=np.float64)
    rr = np.sqrt((yy - float(center_yx[0])) ** 2 + (xx - float(center_yx[1])) ** 2)
    valid = np.isfinite(resid)
    if mask is not None:
        valid &= ~np.asarray(mask, dtype=bool)
    bins = np.floor(rr / float(bin_width_px)).astype(int)
    nbin = int(np.nanmax(bins)) + 1
    profile = np.full(nbin, np.nan, dtype=np.float64)
    radii = (np.arange(nbin, dtype=np.float64) + 0.5) * float(bin_width_px)
    for b in range(nbin):
        pix = valid & (bins == b)
        if np.count_nonzero(pix) >= 3:
            profile[b] = np.nanmedian(resid[pix])
    finite = np.isfinite(profile)
    if np.count_nonzero(finite) >= 2:
        profile[~finite] = np.interp(radii[~finite], radii[finite], profile[finite])
    else:
        profile[~finite] = 0.0
    sigma_bins = max(float(smoothing_scale_px) / float(bin_width_px), 0.0)
    if sigma_bins > 0:
        profile = gaussian_filter1d(profile, sigma=sigma_bins, mode="nearest")
    return radii, profile


def evaluate_radial_profile(shape, center_yx, radii, profile):
    yy, xx = np.indices(shape, dtype=np.float64)
    rr = np.sqrt((yy - float(center_yx[0])) ** 2 + (xx - float(center_yx[1])) ** 2)
    return np.interp(rr.ravel(), np.asarray(radii), np.asarray(profile), left=profile[0], right=profile[-1]).reshape(shape)


def _bad_windows(cfg):
    if cfg.get("drop_wave_min_A") is not None and cfg.get("drop_wave_max_A") is not None:
        return [[float(cfg["drop_wave_min_A"]), float(cfg["drop_wave_max_A"])]]
    return [[5780.0, 6050.0]]


def make_bins(wave, bin_A, bad_windows, min_channels=3):
    lo, hi = float(wave.min()), float(wave.max())
    edges = np.arange(lo, hi + bin_A, bin_A)
    bins = []
    for a, b in zip(edges[:-1], edges[1:]):
        mid = 0.5 * (a + b)
        if any(w0 <= mid <= w1 for w0, w1 in bad_windows):
            continue
        sel = (wave >= a) & (wave < b)
        if sel.sum() >= min_channels:
            bins.append((a, b, mid, sel))
    return bins


def _ring_residual(image, model, companion_yx, mask_radius, width=1.5):
    ny, nx = image.shape
    cy, cx = ny // 2, nx // 2
    yy, xx = np.mgrid[0:ny, 0:nx]
    r = np.hypot(yy - cy, xx - cx)
    comp_r = float(np.hypot(companion_yx[0] - cy, companion_yx[1] - cx))
    comp = np.hypot(yy - companion_yx[0], xx - companion_yx[1])
    ann = (np.abs(r - comp_r) < width) & np.isfinite(image) & (comp > mask_radius)
    halo = np.nanmedian(image[ann])
    if not np.isfinite(halo) or halo == 0:
        return float("nan")
    return float(100.0 * np.nanmedian(np.abs(image[ann] - model[ann])) / halo)


def _as_cube(cube_zyx, name="cube") -> np.ndarray:
    cube = np.asarray(cube_zyx, dtype=np.float64)
    if cube.ndim != 3:
        raise ValueError(f"Expected {name} with shape (nz,ny,nx), got {cube.shape}.")
    return cube


def annulus_background_spectrum(cube_zyx, center_yx, r_in, r_out, *, exclude_yx=None, exclude_radius=0.0):
    """Per-channel local background = median of a source-free annulus.

    Used for the wings-intact aperture-correction path: subtracting a distant
    annulus (rather than a local surface, stage04b) preserves the companion's
    PSF wings so the PSF growth-curve aperture correction stays self-consistent
    (box3<box5). Excludes a region around ``exclude_yx`` (the primary)."""

    cube = _as_cube(cube_zyx)
    _, ny, nx = cube.shape
    yy, xx = np.mgrid[0:ny, 0:nx]
    r = np.hypot(yy - float(center_yx[0]), xx - float(center_yx[1]))
    mask = (r >= float(r_in)) & (r <= float(r_out))
    if exclude_yx is not None and float(exclude_radius) > 0:
        mask &= np.hypot(yy - float(exclude_yx[0]), xx - float(exclude_yx[1])) > float(exclude_radius)
    if not mask.any():
        return np.zeros(cube.shape[0], dtype=np.float64)
    vals = cube[:, mask]
    with np.errstate(all="ignore"):
        return np.nanmedian(vals, axis=1).astype(np.float64)


class VerificationError(RuntimeError):
    """Raised when a requested verification cannot be completed."""


def circular_aperture_mask(shape: tuple[int, int], yx: tuple[float, float], radius: float) -> np.ndarray:
    y, x = np.indices(shape, dtype=np.float64)
    cy, cx = yx
    return (y - cy) ** 2 + (x - cx) ** 2 <= radius**2


def extract_aperture_spectrum(cube: np.ndarray, yx: tuple[float, float], radius: float) -> np.ndarray:
    mask = circular_aperture_mask(cube.shape[1:], yx, radius)
    if not mask.any():
        raise VerificationError("Aperture contains no pixels.")
    return np.nansum(cube[:, mask], axis=1)


def encircled_energy_metric(image, model, center_yx, *, norm_radius_px=25.0, box_half=1,
                            image_background=0.0, model_background=0.0, exclude_mask=None,
                            radii_px=None):
    """V4 de la spec C1: energia encapsulada del MODELO contra la del DATO.

    La metrica del anillo (§3.4) mira el halo en el radio del compañero y es
    ciega al nucleo. Pero el modelo no se usa solo para el halo: C2/C3 lo usan
    para pasar de una caja de 3x3 al flujo dentro de ``norm_radius_px``, y una
    forma puede clavar el anillo con un nucleo completamente equivocado — es lo
    que hace la Moffat cuando el recorte sigma se come el nucleo del ajuste.

    Devuelve el cociente nucleo/norm de los dos, su error relativo, y la curva
    de crecimiento normalizada a ``norm_radius_px`` con su desviacion maxima.
    """

    radii = (np.asarray(radii_px, dtype=np.float64) if radii_px is not None
             else np.asarray([1.0, 2.0, 3.0, 5.0, 8.0, 12.0, 18.0, float(norm_radius_px)]))
    razon_dato = core_to_norm_ratio(image, center_yx, norm_radius_px=norm_radius_px,
                                    box_half=box_half, background=image_background,
                                    exclude_mask=exclude_mask)
    razon_modelo = core_to_norm_ratio(model, center_yx, norm_radius_px=norm_radius_px,
                                      box_half=box_half, background=model_background,
                                      exclude_mask=exclude_mask)
    ee_dato = encircled_energy(image, center_yx, radii, background=image_background,
                               exclude_mask=exclude_mask)
    ee_modelo = encircled_energy(model, center_yx, radii, background=model_background,
                                 exclude_mask=exclude_mask)
    with np.errstate(divide="ignore", invalid="ignore"):
        curva_dato = ee_dato / ee_dato[-1]
        curva_modelo = ee_modelo / ee_modelo[-1]
        error_pct = 100.0 * (razon_modelo / razon_dato - 1.0)
        curva_diff = 100.0 * np.abs(curva_modelo - curva_dato)
    return {
        "radii_px": [float(r) for r in radii],
        "core_ratio_data": float(razon_dato),
        "core_ratio_model": float(razon_modelo),
        "core_ratio_error_pct": float(error_pct),
        "growth_curve_data": [float(v) for v in curva_dato],
        "growth_curve_model": [float(v) for v in curva_modelo],
        "growth_curve_max_abs_diff_pct": float(np.nanmax(curva_diff)) if curva_diff.size else float("nan"),
    }


def encircled_energy(image, center_yx, radii_px, *, background=0.0, exclude_mask=None):
    """``F(r <= radio)`` para cada radio, con el fondo restado.

    ``exclude_mask`` quita pixeles (una fuente de campo dentro del radio, p.ej.)
    y hay que pasarle la MISMA a dato y modelo, o la comparacion no es tal.
    """

    img = np.asarray(image, dtype=np.float64) - float(background)
    yy, xx = np.indices(img.shape, dtype=np.float64)
    cy, cx = map(float, center_yx)
    rr = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2)
    ok = np.isfinite(img)
    if exclude_mask is not None:
        ok &= ~np.asarray(exclude_mask, dtype=bool)
    return np.asarray(
        [float(np.sum(np.where(ok & (rr <= float(r)), img, 0.0))) for r in np.atleast_1d(radii_px)],
        dtype=np.float64,
    )


def core_to_norm_ratio(image, center_yx, *, norm_radius_px=25.0, box_half=1,
                       background=0.0, exclude_mask=None):
    """``F(r <= norm_radius) / F(caja)`` alrededor del centro.

    Es, cifra por cifra, la correccion de apertura que aplican C2/C3: alli sale
    de ``1 / sum(PSF_normalizada * pesos_de_la_caja)`` y la PSF esta normalizada
    a 1 dentro de ``norm_radius_px``, o sea el mismo cociente evaluado sobre el
    modelo. Medirlo tambien sobre el DATO es lo que convierte la correccion de
    apertura en una cantidad verificable en vez de una consecuencia del ajuste.

    La caja se centra en el pixel redondeado, igual para dato y modelo: el
    interes es la diferencia entre los dos, no el valor absoluto al subpixel.
    """

    img = np.asarray(image, dtype=np.float64) - float(background)
    yy, xx = np.indices(img.shape, dtype=np.float64)
    cy, cx = round(float(center_yx[0])), round(float(center_yx[1]))
    ok = np.isfinite(img)
    if exclude_mask is not None:
        ok &= ~np.asarray(exclude_mask, dtype=bool)
    caja = ok & (np.abs(yy - cy) <= int(box_half)) & (np.abs(xx - cx) <= int(box_half))
    total = encircled_energy(image, center_yx, [float(norm_radius_px)],
                             background=background, exclude_mask=exclude_mask)[0]
    box = float(np.sum(np.where(caja, img, 0.0)))
    return float(total / box) if box > 0 else float("nan")


def fixed_radius_grid(norm_radius_px):
    radius = float(norm_radius_px)
    half = int(math.ceil(radius))
    yy, xx = np.mgrid[-half : half + 1, -half : half + 1].astype(np.float64)
    mask = (yy**2 + xx**2) <= radius**2
    return yy, xx, mask


def eval_smoothed_parameter(spec, wavelength_A):
    x = (float(wavelength_A) - float(spec["wave_ref_A"])) / float(spec.get("wave_scale_A", 1000.0))
    coeff = np.asarray(spec["coefficients"], dtype=np.float64)
    return float(np.polynomial.polynomial.polyval(x, coeff))


def smooth_parameter(wavelengths_A, values, *, max_degree=2, wave_ref_A=None, wave_scale_A=1000.0):
    wave = np.asarray(wavelengths_A, dtype=np.float64)
    vals = np.asarray(values, dtype=np.float64)
    good = np.isfinite(wave) & np.isfinite(vals)
    if int(np.count_nonzero(good)) == 0:
        raise ValueError("No finite values to smooth.")
    wave_ref = float(np.nanmedian(wave[good])) if wave_ref_A is None else float(wave_ref_A)
    x = (wave[good] - wave_ref) / float(wave_scale_A)
    y = vals[good]
    best = None
    for deg in range(0, min(int(max_degree), y.size - 1) + 1):
        coeff_high = np.polyfit(x, y, deg)
        pred = np.polyval(coeff_high, x)
        rss = float(np.nansum((y - pred) ** 2))
        k = deg + 1
        aic = y.size * math.log(max(rss / max(y.size, 1), 1e-24)) + 2 * k
        if best is None or aic < best["aic"]:
            best = {"degree": deg, "coeff_high": coeff_high, "rss": rss, "aic": aic}
    coeff_low = best["coeff_high"][::-1].astype(float).tolist()
    return {
        "degree": int(best["degree"]),
        "coefficients": coeff_low,
        "wave_ref_A": wave_ref,
        "wave_scale_A": float(wave_scale_A),
        "model": "polynomial",
    }


def moffat_alpha_from_fwhm(fwhm, beta):
    fwhm = float(fwhm)
    beta = float(beta)
    # A Moffat has finite integral only for beta > 1; a degree-N beta(lambda)
    # polynomial from C1 can extrapolate to beta <= 0 at band edges outside its
    # fit range (seen on the LkCa 15 Moffat fit: 382/3681 channels beta<=0),
    # which sends 2**(1/beta) to an OverflowError and crashes every downstream
    # apcorr. Clamp to the physical floor: a no-op for any healthy PSF (beta>1),
    # and it keeps the aperture correction finite where the model is being
    # extrapolated into the non-normalizable regime.
    if not math.isfinite(beta) or beta < MOFFAT_BETA_FLOOR:
        beta = MOFFAT_BETA_FLOOR
    denom = 2.0 * math.sqrt(max(2.0 ** (1.0 / beta) - 1.0, 1e-12))
    return fwhm / denom


def moffat_norm(params, norm_radius_px=25.0):
    yy, xx, mask = fixed_radius_grid(norm_radius_px)
    profile = moffat_elliptical_profile(
        yy,
        xx,
        params["fwhm_maj"],
        params["fwhm_min"],
        params.get("theta_deg", 0.0),
        params["beta"],
    )
    norm = float(np.nansum(profile[mask]))
    if not np.isfinite(norm) or norm <= 0:
        raise RuntimeError("Invalid Moffat normalization.")
    return norm


def moffat_elliptical_profile(dy, dx, fwhm_maj, fwhm_min, theta_deg, beta):
    """Unit-peak elliptical Moffat profile."""

    dy = np.asarray(dy, dtype=np.float64)
    dx = np.asarray(dx, dtype=np.float64)
    theta = np.deg2rad(float(theta_deg))
    cos_t = np.cos(theta)
    sin_t = np.sin(theta)
    x_rot = dx * cos_t + dy * sin_t
    y_rot = -dx * sin_t + dy * cos_t
    alpha_maj = moffat_alpha_from_fwhm(fwhm_maj, beta)
    alpha_min = moffat_alpha_from_fwhm(fwhm_min, beta)
    rr = (x_rot / alpha_maj) ** 2 + (y_rot / alpha_min) ** 2
    return (1.0 + rr) ** (-float(beta))


def normalized_moffat_psf(dy, dx, params, norm_radius_px=25.0):
    profile = moffat_elliptical_profile(
        dy,
        dx,
        params["fwhm_maj"],
        params["fwhm_min"],
        params.get("theta_deg", 0.0),
        params["beta"],
    )
    return profile / moffat_norm(params, norm_radius_px=norm_radius_px)


@functools.lru_cache(maxsize=16384)
def _psfao_image_cached(x_key, npix, system_name, samp, norm_radius):
    """Build (and cache) the normalised Psfao image for one parameter set.

    Building the Psfao model is an FFT (~6 ms). During per-channel PSF fitting
    (C3/C4) the optimiser evaluates the SAME wavelength/params many times while
    varying only flux/position, so caching the image (keyed on the params, grid
    size, sampling and norm radius) turns hours into minutes. Returns the even
    image, its norm_radius integral, and the grid centre."""

    from maoppy.instrument import muse_nfm, muse_wfm
    from maoppy.psfmodel import Psfao

    system = muse_wfm if str(system_name).lower().endswith("wfm") else muse_nfm
    model = Psfao((npix, npix), system=system, samp=samp)
    # Clip to Psfao's physical bounds (smoothed/interpolated params can drift out
    # of range at edge/gap wavelengths).
    low, high = model.bounds
    eps = 1e-6
    x = [
        float(np.clip(
            xi,
            low[i] + eps if np.isfinite(low[i]) else -np.inf,
            high[i] - eps if np.isfinite(high[i]) else np.inf,
        ))
        for i, xi in enumerate(x_key)
    ]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        img = np.asarray(model(x), dtype=np.float64)  # peak at (npix//2, npix//2)
    c = npix // 2
    gy, gx = np.mgrid[0:npix, 0:npix]
    rr = np.hypot(gy - c, gx - c)
    total = float(np.nansum(img[rr <= float(norm_radius)]))
    if not np.isfinite(total) or total <= 0:
        raise RuntimeError("Psfao normalization within norm_radius failed.")
    return img, total, c


def _psfao_wave_bin_A(model_doc):
    """Grid ``_evaluate_psfao`` snaps lambda to, resolved FROM THE DOCUMENT.

    Order: the grid C1 declared (``psfao_wave_bin_A``) -> the spacing of the
    document's own ``param_table`` -> no snapping at all (0.0, exact lambda).
    There is deliberately no numeric default. The historic one was 50 A, half
    the width of the bins C1 actually fits, so every other channel landed
    between two bins and got its Psfao parameters by linear interpolation --
    and those parameters live in the PSD, where they are degenerate: the
    straight line between two fitted bins leaves the valley, the PSF comes out
    ~1% wrong in the halo, and the near-degenerate psffit design (PSF + PSF +
    plane) turns that into a 13% square wave in the extracted spectrum. See
    `docs/2026-08-12_handoff.md` and `apcorr_debug` sections 14-17.

    A document that carries a ``param_table`` therefore snaps to the width of
    those bins: the parameters exist there and nowhere else. Gaps (bins C1
    rejected) are whole multiples of that width, so the *smallest* spacing is
    the grid -- a median would be inflated by the gaps. A poly-only document
    has no grid to respect: the smoothed polynomial is continuous in lambda, so
    it is evaluated exactly, and if such a document ever needs the FFT cache to
    batch channels it has to declare the grid explicitly.
    """

    declared = model_doc.get("psfao_wave_bin_A")
    if declared is not None:
        wave_bin = float(declared)
        if not np.isfinite(wave_bin) or wave_bin < 0:
            raise ValueError(
                f"psfao_wave_bin_A must be finite and >= 0 (0 = no snapping), got {declared!r}.")
        return wave_bin
    table = model_doc.get("param_table") or {}
    lam = np.unique(np.asarray(table.get("lambda_A", []), dtype=np.float64))
    if lam.size >= 2:
        spacing = float(np.min(np.diff(lam)))
        if np.isfinite(spacing) and spacing > 0:
            return spacing
    return 0.0


def _psfao_grid_npix(model_doc, dy, dx, norm_radius):
    """Tamaño de la rejilla en la que se construye la PSF, INDEPENDIENTE de la posición.

    Dos regímenes, los dos dando un ``npix`` que no depende de dónde esté la
    fuente, para que la caché se reutilice entre estrella, compañero y controles
    a un mismo λ:

    * quien pide la apcorr / la curva de crecimiento manda desplazamientos
      dentro de ``norm_radius`` → rejilla pequeña y rápida;
    * ``psffit`` evalúa sobre la imagen entera; la PSF sólo hace falta sobre la
      región conjunta del ajuste (separación + radio), así que se usa un alcance
      fijo (``psfao_grid_reach_px``, 140 px). 140 px es donde converge el flujo
      del compañero en esta geometría (100 submuestrea el halo AO, ~3 % alto;
      140/180/250 coinciden al 0.2 %). Más allá de la rejilla se muestrea 0, que
      es despreciable.
    """

    max_off = 0.0
    if dy.size:
        max_off = max(float(np.nanmax(np.abs(dy))), float(np.nanmax(np.abs(dx))))
    if max_off <= norm_radius:
        reach = norm_radius
    else:
        reach = max(norm_radius, float(model_doc.get("psfao_grid_reach_px", 140.0)))
    return 2 * (int(np.ceil(reach)) + 2)  # par, la fuente en npix//2


def _psfao_params_at(model_doc, wavelength_A, names=_PSFAO_PARAM_NAMES):
    """The seven Psfao parameters of a document at one wavelength.

    Split out of ``_evaluate_psfao`` because the mixture form needs exactly the
    same resolution rule for each of its components: interpolate the per-bin
    ``param_table`` when there is one (the Psfao PSD parameters are degenerate,
    so smoothing them independently and rebuilding corrupts the PSF), and only
    fall back to ``smoothed_poly`` when the document carries no table.
    """

    table = model_doc.get("param_table")
    if table:
        lam = np.asarray(table["lambda_A"], dtype=np.float64)
        w = float(np.clip(float(wavelength_A), lam.min(), lam.max()))
        return [float(np.interp(w, lam, np.asarray(table[name], dtype=np.float64))) for name in names]
    poly = model_doc.get("smoothed_poly") or {}
    return [
        float(np.polyval(np.asarray(poly[name], dtype=np.float64), float(wavelength_A)))
        for name in names
    ]


def _evaluate_psfao(model_doc, wavelength_A, dy, dx):
    """Evaluate a physical AO PSF (maoppy Psfao) on the (dy, dx) offsets,
    normalised so it sums to 1 within ``norm_radius_px``. Mirrors the Moffat
    branch's contract so aperture-correction/growth-curve/optimal/psffit
    consumers are agnostic to the PSF form. Params come from the C1 per-bin
    ``param_table`` (interpolated) or ``smoothed_poly``. The expensive FFT build
    is cached in ``_psfao_image_cached``; here we only re-sample it."""

    from maoppy.instrument import muse_nfm
    from scipy.ndimage import map_coordinates

    dy = np.asarray(dy, dtype=np.float64)
    dx = np.asarray(dx, dtype=np.float64)
    if dy.shape != dx.shape:
        raise ValueError("psfao evaluation expects matching dy/dx offset arrays.")
    # FWHM perturbation (C3/E4 sensitivity tests). The Psfao parameters live in
    # the PSD, so there is no coefficient to multiply the way the Moffat branch
    # does: the geometric equivalent is to sample the built PSF on offsets
    # divided by the scale (a dilation by `scale`), with 1/scale**2 conserving
    # the integral. See `scaled_psf_model`.
    fwhm_scale = float(model_doc.get("psf_fwhm_scale", 1.0))
    if not np.isfinite(fwhm_scale) or fwhm_scale <= 0:
        raise ValueError(f"psf_fwhm_scale must be finite and > 0, got {fwhm_scale!r}.")
    if fwhm_scale != 1.0:
        dy = dy / fwhm_scale
        dx = dx / fwhm_scale
    names = model_doc.get("param_names", _PSFAO_PARAM_NAMES)
    # Snap the wavelength to the grid the parameters were fitted on before
    # building the PSF: consecutive channels then share ONE cached FFT build
    # (3681 builds -> ~45), and -- the reason the grid must not be finer than
    # C1's bins -- no channel gets its degenerate PSD parameters from a point
    # halfway between two fits. `_psfao_wave_bin_A` resolves it from the
    # document; 0 means evaluate at the exact wavelength. Sampling always uses
    # the exact per-call offsets, so per-channel positions/flux stay exact.
    wave_bin = _psfao_wave_bin_A(model_doc)
    w_eff = round(float(wavelength_A) / wave_bin) * wave_bin if wave_bin > 0 else float(wavelength_A)
    x = _psfao_params_at(model_doc, w_eff, names)
    system_name = "muse_wfm" if str(model_doc.get("system", "muse_nfm")).lower().endswith("wfm") else "muse_nfm"
    samp = float(muse_nfm.samp(w_eff * 1e-10))
    norm_radius = float(model_doc.get("norm_radius_px", 25.0))
    npix = _psfao_grid_npix(model_doc, dy, dx, norm_radius)
    img, total, c = _psfao_image_cached(
        tuple(round(v, 10) for v in x), npix, system_name, round(samp, 10), round(norm_radius, 6)
    )
    rows = (c + dy).ravel()
    cols = (c + dx).ravel()
    vals = map_coordinates(img, [rows, cols], order=1, mode="constant", cval=0.0).reshape(dy.shape)
    return vals / (total * fwhm_scale ** 2)


def _mixture_component_key(component, wavelength_A):
    """La componente, reducida a algo hasheable: (forma, parámetros a ese λ).

    La clave es lo que hace cacheable la imagen de la mezcla. Es el mismo truco
    que ``_psfao_image_cached`` usa con los siete parámetros, extendido a N
    componentes: dos llamadas al mismo λ producen la misma clave y la suma
    pesada no se vuelve a construir.
    """

    model = component["model"]
    form = str(model.get("form", "moffat")).lower()
    if form == "psfao":
        names = tuple(model.get("param_names", _PSFAO_PARAM_NAMES))
        values = _psfao_params_at(model, wavelength_A, names)
        return ("psfao", tuple(round(float(v), 10) for v in values))
    if form == "moffat":
        params = {
            key: float(eval_smoothed_parameter(model["coefficients"][key], wavelength_A))
            for key in PSF_SHAPE_PARAMS
        }
        return ("moffat", tuple(round(params[key], 10) for key in PSF_SHAPE_PARAMS))
    raise ValueError(
        f"Mixture component has form={form!r}; expected 'moffat' or 'psfao'."
    )


def _mixture_component_weight(component, wavelength_A):
    """Peso de una componente a ese λ: peso del combinado × flujo de la exposición.

    El combinado es una media **pesada de brillo**, así que la PSF del combinado
    es la media de las PSF pesada por `w_i · F_i(λ)`, no por `w_i` sola: la
    transmisión y la masa de aire cambian entre exposiciones y el brillo de la
    estrella con ellas. `F_i` es el flujo dentro de `norm_radius_px`, que es la
    región en la que cada componente está normalizada, y lo mide C1 cuando
    construye el documento — aquí no se re-deriva.
    """

    weight = float(component.get("weight", 1.0))
    flux = component.get("flux_norm") or {}
    lam = np.asarray(flux.get("lambda_A", ()), dtype=np.float64)
    values = np.asarray(flux.get("value", ()), dtype=np.float64)
    if lam.size == 0 or values.size != lam.size:
        raise ValueError(
            f"Mixture component {component.get('exposure_id', '?')!r} has no usable "
            "`flux_norm`; the mixture weight is w_i*F_i(lambda) and F_i cannot be guessed."
        )
    ok = np.isfinite(lam) & np.isfinite(values) & (values > 0)
    if not ok.any():
        raise ValueError(
            f"Mixture component {component.get('exposure_id', '?')!r} has no finite "
            "positive `flux_norm` values."
        )
    lam_ok, values_ok = lam[ok], values[ok]
    w = float(np.clip(float(wavelength_A), lam_ok.min(), lam_ok.max()))
    return weight * float(np.interp(w, lam_ok, values_ok))


@functools.lru_cache(maxsize=256)
def _mixture_image_cached(keys, weights, npix, system_name, samp, norm_radius):
    """Imagen de la mezcla, normalizada a 1 dentro de ``norm_radius``.

    Se cachea la SUMA, no sólo las componentes: sin esto, cada evaluación de la
    PSF costaría N veces una evaluación normal (con 29 exposiciones, el ajuste
    por canal de C4 pasaría de minutos a horas). Con la suma cacheada, una
    mezcla cuesta lo mismo que una PSF suelta salvo la primera vez a cada λ, y
    esa primera vez son N construcciones (~0.35 s cada una en la rejilla grande
    de psffit, ~5 ms en la pequeña de la apcorr).

    Las componentes se construyen **saltándose** ``_psfao_image_cached``: al
    estar la suma cacheada sólo hacen falta una vez por λ, y dejarlas en esa
    caché guardaría 29×43 imágenes de 284² (~800 MB) que nadie volvería a
    mirar. Lo que se retiene es la mezcla, que son ~28 MB.
    """

    build_component = getattr(_psfao_image_cached, "__wrapped__", _psfao_image_cached)
    c = npix // 2
    gy, gx = np.mgrid[0:npix, 0:npix]
    dy = gy - c
    dx = gx - c
    stack = np.zeros((npix, npix), dtype=np.float64)
    for (kind, params), weight in zip(keys, weights):
        if kind == "psfao":
            img, total, _ = build_component(params, npix, system_name, samp, norm_radius)
            unit = img / total
        else:
            unit = normalized_moffat_psf(
                dy, dx, dict(zip(PSF_SHAPE_PARAMS, params)), norm_radius_px=norm_radius
            )
        stack += float(weight) * unit
    inside = np.hypot(dy, dx) <= float(norm_radius)
    total = float(np.nansum(stack[inside]))
    if not np.isfinite(total) or total <= 0:
        raise RuntimeError("Mixture normalization within norm_radius failed.")
    return stack / total, c


def _evaluate_mixture(model_doc, wavelength_A, dy, dx):
    """Evalúa una PSF de mezcla: la media pesada de las PSF por observación.

    Existe porque **la mezcla de N PSF no es una PSF**: el combinado suma
    exposiciones con seeing y calidad de AO distintas, y ninguna Psfao ni
    Moffat puede describir esa suma. Ajustar una forma analítica al combinado es
    lo que dejaba la razón núcleo/halo —y con ella la corrección de apertura—
    sistemáticamente mal. Aquí cada exposición aporta su propio modelo, ajustado
    a su propio cubo, y la suma se hace con los pesos del combinado.

    El contrato es el mismo que el de las otras dos formas: devuelve la PSF
    normalizada a 1 dentro de ``norm_radius_px``, así que quien la consuma
    (apcorr, curva de crecimiento, optimal, psffit, inyección) no necesita
    saber que es una mezcla.
    """

    from maoppy.instrument import muse_nfm
    from scipy.ndimage import map_coordinates

    dy = np.asarray(dy, dtype=np.float64)
    dx = np.asarray(dx, dtype=np.float64)
    if dy.shape != dx.shape:
        raise ValueError("mixture evaluation expects matching dy/dx offset arrays.")
    components = model_doc.get("components") or ()
    if not components:
        raise ValueError("Mixture psf_model has no components.")

    # Misma mecánica geométrica que la rama psfao (ver `scaled_psf_model`): la
    # escala de FWHM se aplica a la mezcla entera, no componente a componente,
    # porque una dilatación conmuta con la suma pesada.
    fwhm_scale = float(model_doc.get("psf_fwhm_scale", 1.0))
    if not np.isfinite(fwhm_scale) or fwhm_scale <= 0:
        raise ValueError(f"psf_fwhm_scale must be finite and > 0, got {fwhm_scale!r}.")
    if fwhm_scale != 1.0:
        dy = dy / fwhm_scale
        dx = dx / fwhm_scale

    wave_bin = _psfao_wave_bin_A(model_doc)
    w_eff = round(float(wavelength_A) / wave_bin) * wave_bin if wave_bin > 0 else float(wavelength_A)
    keys = tuple(_mixture_component_key(component, w_eff) for component in components)
    raw = np.asarray(
        [_mixture_component_weight(component, w_eff) for component in components],
        dtype=np.float64,
    )
    total_weight = float(np.nansum(raw))
    if not np.isfinite(total_weight) or total_weight <= 0:
        raise RuntimeError(f"Mixture weights are not usable at {wavelength_A} A.")
    # Normalizados antes de redondear: así la clave de la caché no depende de la
    # escala de flujo absoluta, que cambia con λ aunque la mezcla sea la misma.
    weights = tuple(round(float(v / total_weight), 9) for v in raw)

    system_name = (
        "muse_wfm"
        if str(model_doc.get("system", "muse_nfm")).lower().endswith("wfm")
        else "muse_nfm"
    )
    samp = float(muse_nfm.samp(w_eff * 1e-10))
    norm_radius = float(model_doc.get("norm_radius_px", 25.0))
    npix = _psfao_grid_npix(model_doc, dy, dx, norm_radius)
    img, c = _mixture_image_cached(
        keys, weights, npix, system_name, round(samp, 10), round(norm_radius, 6)
    )
    rows = (c + dy).ravel()
    cols = (c + dx).ravel()
    vals = map_coordinates(img, [rows, cols], order=1, mode="constant", cval=0.0).reshape(dy.shape)
    return vals / (fwhm_scale ** 2)


def evaluate_psf_model(model_doc, wavelength_A, dy, dx):
    form = str(model_doc.get("form", "moffat")).lower()
    if form == "psfao":
        return _evaluate_psfao(model_doc, wavelength_A, dy, dx)
    if form == MIXTURE_FORM:
        return _evaluate_mixture(model_doc, wavelength_A, dy, dx)
    if form != "moffat":
        raise ValueError(
            f"Unsupported psf_model form={form!r}; expected 'moffat', 'psfao' or 'mixture'."
        )
    params = {
        key: eval_smoothed_parameter(model_doc["coefficients"][key], wavelength_A)
        for key in PSF_SHAPE_PARAMS
    }
    return normalized_moffat_psf(
        dy,
        dx,
        params,
        norm_radius_px=float(model_doc.get("norm_radius_px", 25.0)),
    )


In [ ]:
import ast as _ast, hashlib as _hashlib

_SHAS = {
    "musepipe/psf.py:source_mask": "21b7a974898a",
    "musepipe/psf.py:companion_ring_metric": "5b37f4cb618a",
    "musepipe/psf.py:radial_hybrid_profile": "24c26c3330fd",
    "musepipe/psf.py:evaluate_radial_profile": "2c72e505513d",
    "musepipe/stages/stage_e01_psfao.py:_bad_windows": "5a64b57438d1",
    "musepipe/stages/stage_e01_psfao.py:make_bins": "26d30a7ccd07",
    "musepipe/stages/stage_e01_psfao.py:_ring_residual": "8392c32f6e2f",
    "musepipe/extraction/aperture.py:_as_cube": "67ede036d305",
    "musepipe/extraction/aperture.py:annulus_background_spectrum": "68ec4c4e7299",
    "musepipe/reduction/verify.py:VerificationError": "d9acb4457343",
    "musepipe/reduction/verify.py:circular_aperture_mask": "b08d990cd3a2",
    "musepipe/reduction/verify.py:extract_aperture_spectrum": "e46a9616dd50",
    "musepipe/psf.py:encircled_energy_metric": "ea9427d959c5",
    "musepipe/psf.py:encircled_energy": "776b7d01b66e",
    "musepipe/psf.py:core_to_norm_ratio": "1d40d2049c41",
    "musepipe/psf.py:fixed_radius_grid": "f78dc6842678",
    "musepipe/psf.py:eval_smoothed_parameter": "05d5d9150afb",
    "musepipe/psf.py:smooth_parameter": "a255fe5c283a",
    "musepipe/psf.py:moffat_alpha_from_fwhm": "8706bfcfbc81",
    "musepipe/psf.py:moffat_norm": "70e93f0c9d58",
    "musepipe/psf.py:moffat_elliptical_profile": "eb8198bdfcf6",
    "musepipe/psf.py:normalized_moffat_psf": "25a3c40a75af",
    "musepipe/psf.py:_psfao_image_cached": "48bc78ca330c",
    "musepipe/psf.py:_psfao_wave_bin_A": "fa190a8b7628",
    "musepipe/psf.py:_psfao_grid_npix": "3a7f484ee3dd",
    "musepipe/psf.py:_psfao_params_at": "1475d730e811",
    "musepipe/psf.py:_evaluate_psfao": "97cbdbedc4a6",
    "musepipe/psf.py:_mixture_component_key": "8dd6f1c31b65",
    "musepipe/psf.py:_mixture_component_weight": "fffa99de93df",
    "musepipe/psf.py:_mixture_image_cached": "73da0a34babb",
    "musepipe/psf.py:_evaluate_mixture": "c7b276a3dddd",
    "musepipe/psf.py:evaluate_psf_model": "afb0201f33a3",
    "musepipe/psf.py:MOFFAT_BETA_FLOOR": "fc34677ec806",
    "musepipe/psf.py:_PSFAO_PARAM_NAMES": "d27321df07fa",
    "musepipe/psf.py:PSF_SHAPE_PARAMS": "bbf6c549da2c",
    "musepipe/psf.py:MIXTURE_FORM": "636b1828b300"
}

def _pieza(cuerpo, name):
    """El nodo que define `name`: def/class, o la asignación de una constante.

    Las constantes también se vigilan: viajan copiadas igual que las
    funciones, y hasta ahora nadie comprobaba que siguieran siendo las de
    `musepipe` — añadir una banda a un diccionario dejaba esta copia atrás
    sin que nada lo dijera.
    """
    for n in cuerpo:
        if isinstance(n, (_ast.FunctionDef, _ast.ClassDef)) and n.name == name:
            inicio = min([n.lineno] + [d.lineno for d in n.decorator_list])
            return inicio, n.end_lineno
        if isinstance(n, _ast.Assign) and any(
                isinstance(t, _ast.Name) and t.id == name for t in n.targets):
            return n.lineno, n.end_lineno
        if (isinstance(n, _ast.AnnAssign) and isinstance(n.target, _ast.Name)
                and n.target.id == name):
            return n.lineno, n.end_lineno
    return None

def chequeo_de_deriva(shas=_SHAS, root=ROOT):
    problemas = []
    for key, sha in shas.items():
        rel, name = key.rsplit(':', 1)
        text = (root / rel).read_text(encoding='utf-8')
        lines = text.splitlines(keepends=True)
        sitio = _pieza(_ast.parse(text).body, name)
        if sitio is None:
            problemas.append(f'{key}: ya no existe en musepipe'); continue
        inicio, fin = sitio
        src = ''.join(lines[inicio - 1:fin]).rstrip('\n')
        actual = _hashlib.sha256(src.encode('utf-8')).hexdigest()[:12]
        if actual != sha:
            problemas.append(f'{key}: la copia es {sha}, musepipe tiene {actual}')
    return problemas

_deriva = chequeo_de_deriva()
if _deriva:
    print('DERIVA — la cadena cambió y esta copia se quedó atrás:')
    for p in _deriva:
        print('  ·', p)
    print(f'\nRegenera: python scripts/build_debug_notebooks.py --target {TARGET} PSFHALO')
else:
    print(f'sin deriva: las {len(_SHAS)} piezas copiadas son las de musepipe')


### 2.b · Anclaje contra la cadena

Dos comprobaciones exactas antes de contar nada.


In [ ]:
yy, xx = np.mgrid[-60:61, -60:61].astype(float)
_csv = SD / 'stage_e01_psfao_params.csv'
_filas = list(csv.DictReader(open(_csv))) if _csv.exists() else []
_lam_bins = [float(r['lambda_A']) for r in _filas]

# El QC resume el anillo por bin con una mediana. Reproducirla desde el
# CSV es exacto y no toca el cubo: si coincide, este notebook lee las
# mismas medidas que la cadena publica.
def _mediana(col):
    v = [float(r[col]) for r in _filas
         if r.get(col) not in (None, '', 'nan')]
    return float(np.median(v)) if v else float('nan')

_suyo = float(QC_C1['companion_ring_metric']['residual_pct_median'])
_mio = _mediana('ring_residual_pct_after_hybrid_canonical')
_sin = _mediana('ring_residual_pct_canonical')
_ok = bool(np.isfinite(_mio) and abs(_mio - _suyo) <= 1e-9)
print(f'anillo del QC          : {_suyo:.9f} %')
print(f'reconstruido del CSV   : {_mio:.9f} %   sobre {len(_filas)} bins')
print(f'   |Δ| = {abs(_mio - _suyo):.2e}')

print(f'\ny el mismo anillo SIN el término híbrido: {_sin:.3f} %')
_hyb_comp = {str(c['model'].get('hybrid')) for c in
             (PSF_ENTREGADO.get('components') or [])} or {'n/a'}
print(f'   el modelo entregado lo lleva: {", ".join(sorted(_hyb_comp))}')
print('   -> el QC titula con el número CON híbrido; el modelo que consumen')
print('      C2-C6 es el otro. La §8 vuelve sobre esto.')

_ph = SD / 'psf_hybrid_residual.fits'
if _ph.exists():
    with fits.open(_ph) as _h:
        print(f'\npsf_hybrid_residual.fits: {_h["PROFILE"].data.shape[0]} bins'
              f' × {_h["RADIUS_PX"].data.size} radios')
        _ok &= _h['PROFILE'].data.shape[0] == len(_filas)
print()
print('IDÉNTICO: la copia reproduce la cadena.' if _ok else
      'DIFIERE — si has tocado una perilla, es lo esperado;'
      ' si no, mira el chequeo de deriva.')


## 3 · Para qué usa la cadena la PSF, y por qué importa dónde falle

El modelo tiene **dos usos distintos**, y el error no les afecta igual:

**Uso 1 — la corrección de apertura.** El compañero es débil, así que se mide en un cuadro de 3×3 px, que maximiza la S/N pero recoge solo una fracción de su luz. Esa fracción no se puede medir en el compañero, así que sale del modelo:

$$\mathrm{apcorr} = \frac{F(\le R_{\rm norm})}{F(\mathrm{box3})}$$

Solo depende del modelo **dentro** de $R_{\rm norm}$. El tramo de ahí hacia fuera lo mide A2 empíricamente sobre la primaria, así que el flujo publicado es total.

**Uso 2 — el fondo espacial.** `psffit` ajusta la primaria y el objeto a la vez, y `optimal_psfsub` resta el modelo de la primaria. Los dos usan el modelo **fuera** del núcleo, donde vive el halo.

> **De ahí la consecuencia que ordena todo lo demás:** un error en el núcleo mueve el flujo de *todos* los métodos; un error en el halo **solo** muerde a los dos que restan el modelo. La §9 lo confirma con números.


In [ ]:
_lam0 = float(np.median(_lam_bins)) if _lam_bins else 6550.0
_m = evaluate_psf_model(PSF_ENTREGADO, _lam0, yy, xx)
_r = np.hypot(yy, xx)
_rr = np.arange(1.0, 60.0, 0.5)
_enc = np.array([np.nansum(_m[_r <= q]) for q in _rr])
_enc = _enc / np.nansum(_m[_r <= R_NORM])
fig, ax = plt.subplots(figsize=(7.2, 3.6))
ax.plot(_rr, _enc, lw=2)
ax.axvline(R_NORM, color='crimson', ls='--',
           label=f'$R_{{\\rm norm}}$ = {R_NORM:.0f} px  ·  aquí manda el USO 1')
ax.axvspan(R_NORM, 60, color='0.85', alpha=.5,
           label='fuera: el halo, USO 2 (y A2 mide el flujo)')
ax.axvline(1.5, color='navy', ls=':', label='box3 (medio lado 1.5 px)')
ax.set_xlabel('radio [px]'); ax.set_ylabel(r'$F(\leq r)\,/\,F(\leq R_{\rm norm})$')
ax.set_title('los dos usos del modelo, sobre la misma curva de crecimiento')
ax.legend(fontsize=8); ax.grid(alpha=.3); plt.show()
_frac_box3 = float(np.nansum(_m[(np.abs(yy) <= 1.5) & (np.abs(xx) <= 1.5)])
                   / np.nansum(_m[_r <= R_NORM]))
print(f'a λ={_lam0:.0f} Å el modelo pone en box3 el {100 * _frac_box3:.1f}% de la luz'
      f' de dentro de {R_NORM:.0f} px  ->  apcorr = {1 / _frac_box3:.2f}')
print(f'y el QC declara un error de ese cociente de'
      f' {QC_C1["encircled_energy"]["core_ratio_error_pct_median"]:.2f} %'
      f' (tolerancia {QC_C1["encircled_energy"]["tolerance_pct"]:.0f} %)')


## 4 · Dónde falla el modelo, y cuánto

Se compara el **perfil radial** del dato con el del modelo, en seis bandas de λ, normalizando el modelo al dato dentro de $R_{\rm norm}$ — que es la convención con la que la cadena lo usa.

> **Un aviso que costó una medida.** El cubo del que extrae la cadena está **recortado**, y la posición de la primaria que publica B3 está en *ese* marco. Usar esa posición sobre el cubo sin recortar desplaza el centro ~21 px y el perfil sale mal. Aquí se lee el cubo de la cadena y **se verifica el centro contra el pico de brillo** antes de medir nada.


In [ ]:
QC_B3 = json.loads((SD / 'stage01c_qc.json').read_text(encoding='utf-8'))
SY, SX = map(float, QC_B3['primary']['pos_yx'])
CY, CX = map(float, QC_B3['companion']['pos_yx'])
SEP = float(np.hypot(CY - SY, CX - SX))
with fits.open(SD / 'stage02_xcorr_cube_stack.fits', memmap=True) as _h:
    _lam = np.asarray(_h['WAVELENGTH'].data, float)
    IMGS, LAMS = [], []
    for _a, _b in BANDAS:
        _j = np.where((_lam >= _a) & (_lam < _b))[0]
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            IMGS.append(np.nanmedian(_h['CUBES'].data[0, _j[0]:_j[-1] + 1],
                                     axis=0).astype(float))
        LAMS.append(float(np.median(_lam[_j])))
NY, NX = IMGS[0].shape
YY, XX = np.mgrid[0:NY, 0:NX].astype(float)
DY, DX = YY - SY, XX - SX
RR = np.hypot(DY, DX)
_pk = np.unravel_index(np.nanargmax(np.where(np.isfinite(IMGS[2]), IMGS[2], -np.inf)),
                       IMGS[2].shape)
print(f'cubo {NY}×{NX} · primaria declarada ({SY:.2f}, {SX:.2f})'
      f' · pico de brillo {_pk}')
print(f'   desviación {np.hypot(_pk[0] - SY, _pk[1] - SX):.2f} px'
      f'  ->  {"el marco es el correcto" if np.hypot(_pk[0] - SY, _pk[1] - SX) < 2 else "MARCO EQUIVOCADO"}')
print(f'   compañero a {SEP:.2f} px de la primaria')


In [ ]:
def perfil_residuo(img, modelo, anclaje=None, radios=RADIOS_MUESTRA):
    """(dato - modelo)/modelo en anillos de +-2 px, con el modelo
    normalizado al dato dentro de `anclaje` (por defecto R_NORM)."""
    anclaje = R_NORM if anclaje is None else anclaje
    _s = (RR <= anclaje) & np.isfinite(img) & np.isfinite(modelo)
    m = modelo * (np.nansum(img[_s]) / np.nansum(modelo[_s]))
    out = []
    for q in radios:
        an = (RR >= q - 2) & (RR < q + 2) & np.isfinite(img) & np.isfinite(m)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            out.append(100 * (np.nanmedian(img[an]) - np.nanmedian(m[an]))
                       / abs(np.nanmedian(m[an])))
    return np.asarray(out), m

# El modelo ENTREGADO es el que consumen C2-C6: es el que hay que auditar.
MODELOS = [evaluate_psf_model(PSF_ENTREGADO, L, DY, DX) for L in LAMS]
TABLA = np.asarray([perfil_residuo(i, m)[0] for i, m in zip(IMGS, MODELOS)])
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.5, 4.0))
for k, L in enumerate(LAMS):
    a1.plot(RADIOS_MUESTRA, TABLA[k], 'o-', label=f'{L:.0f} Å')
a1.axhline(0, color='k', lw=.8); a1.axvline(R_NORM, color='crimson', ls='--')
a1.axvline(SEP, color='navy', ls=':', label='compañero')
a1.set_xlabel('radio [px]'); a1.set_ylabel('(dato − modelo) / modelo  [%]')
a1.set_title('el error del modelo crece con el radio'); a1.legend(fontsize=7)
a1.grid(alpha=.3)
_im = a2.imshow(TABLA, aspect='auto', origin='lower', cmap='RdBu_r',
                vmin=-np.nanmax(np.abs(TABLA)), vmax=np.nanmax(np.abs(TABLA)),
                extent=[RADIOS_MUESTRA[0], RADIOS_MUESTRA[-1], 0, len(LAMS)])
a2.set_yticks(np.arange(len(LAMS)) + .5)
a2.set_yticklabels([f'{L:.0f}' for L in LAMS])
a2.set_xlabel('radio [px]'); a2.set_ylabel('λ [Å]')
a2.set_title('y cambia con λ: el problema es cromático')
plt.colorbar(_im, ax=a2, label='residuo [%]'); plt.tight_layout(); plt.show()
_k = int(np.argmin([abs(L - 6600) for L in LAMS]))
print(f'en la banda de {LAMS[_k]:.0f} Å el residuo va de'
      f' {TABLA[_k][0]:+.1f} % a {RADIOS_MUESTRA[0]:.0f} px'
      f' a {TABLA[_k][-1]:+.1f} % a {RADIOS_MUESTRA[-1]:.0f} px')
_cruce = [RADIOS_MUESTRA[i] for i in range(1, len(RADIOS_MUESTRA))
          if TABLA[_k][i - 1] < 0 <= TABLA[_k][i]]
print(f'   cruza cero cerca de {_cruce[0]:.0f} px' if _cruce else
      '   no cruza cero en el rango medido')


### 4.a · El mismo residuo, en dos dimensiones

El perfil radial promedia sobre ángulo. El mapa 2D enseña lo que ese promedio esconde, y distingue tres cosas que un número no puede: un **anillo** a radio fijo (la forma está mal), una **falda suave** (falta halo) o un **pedestal plano** (sería problema del dato, no del modelo).


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12.5, 4.2))
for ax, k in zip(axes, (0, 2, 5)):
    _s = (RR <= R_NORM) & np.isfinite(IMGS[k]) & np.isfinite(MODELOS[k])
    _m = MODELOS[k] * (np.nansum(IMGS[k][_s]) / np.nansum(MODELOS[k][_s]))
    _res = (IMGS[k] - _m)
    _esc = np.nanpercentile(np.abs(_res[RR > R_NORM]), 98)
    ax.imshow(_res, origin='lower', cmap='RdBu_r', vmin=-_esc, vmax=_esc)
    for q in (R_NORM, SEP):
        ax.add_patch(plt.Circle((SX, SY), q, fill=False, color='k', lw=.8, ls='--'))
    ax.plot(CX, CY, 'x', color='lime', ms=9, mew=2)
    ax.set_title(f'{LAMS[k]:.0f} Å'); ax.set_xticks([]); ax.set_yticks([])
fig.suptitle('residuo dato − modelo · círculos: $R_{\\rm norm}$ y el compañero (×)')
plt.tight_layout(); plt.show()
print('lectura: falda extendida y positiva fuera del núcleo, no un pedestal plano')
print('  -> falta HALO, y falta más cuanto más lejos')


## 5 · ¿Lo crea la combinación de exposiciones? No: la atenúa

La sospecha natural era que el problema lo creara **combinar**: el cubo de la cadena promedia decenas de exposiciones con calidad de AO distinta, y una sola forma analítica no puede describir esa mezcla.

**Se midió exposición a exposición** —cada una con *su* cubo y *su* modelo— y la hipótesis se cae: las exposiciones individuales fallan **más**, no menos.


In [ ]:
if not CSV_POR_EXPOSICION.exists():
    print(f'sin {CSV_POR_EXPOSICION.name}: esta sección necesita la tabla'
          ' precalculada (ver el encabezado del CSV para regenerarla).')
else:
    _f = list(csv.DictReader(open(CSV_POR_EXPOSICION)))
    _noches = sorted({r['noche'] for r in _f})
    _cols = [c for c in _f[0] if c.startswith('res_')]
    _rad = np.array([float(c.split('_')[1]) for c in _cols])
    fig, (b1, b2) = plt.subplots(1, 2, figsize=(11.5, 4.0))
    _col = dict(zip(_noches, ('#1f77b4', '#d62728', '#2ca02c')))
    for r in _f:
        b1.plot(_rad, [float(r[c]) for c in _cols], '-', lw=.9, alpha=.65,
                color=_col[r['noche']])
    b1.plot(RADIOS_MUESTRA, TABLA[2], 'k--o', lw=2.2, label='el COMBINADO')
    for n in _noches:
        b1.plot([], [], color=_col[n], label=f'exposiciones de {n}')
    b1.axhline(0, color='k', lw=.8); b1.axvline(SEP, color='navy', ls=':')
    b1.set_xlabel('radio [px]'); b1.set_ylabel('residuo [%]')
    b1.set_title('cada exposición contra SU modelo'); b1.legend(fontsize=7)
    b1.grid(alpha=.3)
    _w = np.array([float(r['fwhm_proxy']) for r in _f])
    _y = np.array([float(r[_cols[-1]]) for r in _f])
    for n in _noches:
        _s = np.array([r['noche'] == n for r in _f])
        b2.plot(_w[_s], _y[_s], 'o', color=_col[n], label=n)
    b2.set_xlabel('anchura del núcleo  $w_{1/2}$ [px]')
    b2.set_ylabel(f'residuo a {_rad[-1]:.0f} px [%]')
    b2.set_title('cuanto más nítido el núcleo, peor el halo')
    b2.legend(fontsize=7); b2.grid(alpha=.3)
    plt.tight_layout(); plt.show()
    print(f'combinado a {_rad[-1]:.0f} px: {TABLA[2][-1]:+.1f} %')
    for n in _noches:
        _v = [float(r[_cols[-1]]) for r in _f if r['noche'] == n]
        print(f'   noche {n} (n={len(_v):2d}): mediana {np.median(_v):+6.1f} %'
              f'  ·  rango {min(_v):+.1f} a {max(_v):+.1f}')
    print(f'\ncorr(anchura del núcleo, residuo a {_rad[-1]:.0f} px)'
          f' = {np.corrcoef(_w, _y)[0, 1]:+.3f}  (n={len(_f)})')
    print('  -> normalizar dentro del núcleo fija la amplitud con él, y con AO buena')
    print('     esos píxeles casi no contienen halo: el halo queda subestimado.')


## 6 · ¿Es el radio de normalización? No

Si el modelo estuviera bien de forma y solo mal de escala, **anclarlo en otro radio** lo arreglaría: sobraría con normalizar donde se mide. Se prueba con tres anclajes.


In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 3.8))
_k = 2
for _anc in ANCLAJES_PRUEBA:
    _p, _ = perfil_residuo(IMGS[_k], MODELOS[_k], anclaje=_anc)
    ax.plot(RADIOS_MUESTRA, _p, 'o-', label=f'anclado en ≤ {_anc:.0f} px')
    print(f'anclaje ≤{_anc:4.0f} px  ->  residuo a {RADIOS_MUESTRA[-1]:.0f} px ='
          f' {_p[-1]:+7.1f} %')
ax.axhline(0, color='k', lw=.8)
ax.set_xlabel('radio [px]'); ax.set_ylabel('residuo [%]')
ax.set_title('mover el anclaje NO arregla la forma'); ax.legend(fontsize=8)
ax.grid(alpha=.3); plt.show()
print('\nlas tres curvas son casi la misma: si fuera un problema de escala, una de')
print('ellas sería plana. El desajuste es de FORMA.')
print('Y ojo: `norm_radius_px` no es un detalle interno — define qué significa el')
print('flujo entregado, porque apcorr = F(<=norm_radius)/F(box3).')


## 7 · ¿Es la ponderación del ajuste? Tampoco

El ajuste por exposición usa una ponderación **propia** (`psf_perobs_fit_weighting`), distinta de la del combinado, y no por descuido: está elegida midiendo. En una exposición suelta el halo está dominado por el moteado de la AO residual, y llevar la atención del ajuste hacia el radio grande **empeora** el resultado.

La tabla que lo decidió vive en el propio código (`stage_e01_perobs.py`), y la conclusión relevante aquí es doble: **ya está en su mejor opción de las tres**, y **lo que el peso decide es el halo, no el núcleo** — el cociente núcleo/total sale bien con las tres.


In [ ]:
from musepipe.stages.stage_e01_perobs import (PEROBS_DEFAULT_WEIGHTING,
                                              PEROBS_DEFAULT_WEIGHT_CAP)
print(f'ajuste al COMBINADO     : weighting={QC_C1["fit"]["weighting"]!r}'
      f'  cap={QC_C1["fit"]["weight_cap"]}')
_po = QC_C1.get('per_observation') or {}
if _po.get('exposures'):
    _w = {(e.get('weighting'), e.get('weight_cap')) for e in _po['exposures']}
    for _ww, _cc in sorted(_w, key=str):
        print(f'ajuste POR EXPOSICIÓN   : weighting={_ww!r}  cap={_cc}'
              f'   ({len(_po["exposures"])} exposiciones)')
    _v4 = _po.get('v4_per_exposure_pct') or {}
    if _v4:
        print(f'\ncociente núcleo/total por exposición: mediana'
              f' {_v4["median"]:.2f} %  (min {_v4["min"]:.2f}, max {_v4["max"]:.2f})')
    _an = [e['ring_residual_pct_median'] for e in _po['exposures']
           if isinstance(e.get('ring_residual_pct_median'), dict)]
    if _an:
        _f2 = _po['exposures'][0]['form_chosen']
        _vals = [a[_f2] for a in _an if _f2 in a]
        print(f'residuo de ANILLO por exposición    : mediana'
              f' {np.median(_vals):.2f} %  ({len(_vals)} exposiciones)')
        print(f'\n  -> el modelo acierta el núcleo y falla el halo:'
              f' factor ~{np.median(_vals) / max(_v4["median"], 1e-9):.0f}'
              ' entre los dos errores')
else:
    print('este run no ajustó por exposición: la sección no aplica.')


## 8 · El término híbrido: qué arregla, qué rompía, y cuánto es real

Si la forma analítica no puede con el halo, la alternativa es **no pedírselo**: se mide lo que le falta y se le suma. Eso es el híbrido — la **mediana azimutal** del residuo, suavizada, sumada al modelo.

Estaba descartado porque **rompía la corrección de apertura**. Y la causa era *dónde* se aplicaba: el perfil se sumaba en **todos** los radios, incluido dentro de $R_{\rm norm}$, así que cambiaba $F(\le R_{\rm norm})$ y con ella el cociente que C2/C3 invierten.

**Restringido a $r > R_{\rm norm}$**, ese daño es *exactamente* nulo por geometría: lo que se suma vale cero justo en la región que define la corrección de apertura.


In [ ]:
_k = 2
_s = (RR <= R_NORM) & np.isfinite(IMGS[_k]) & np.isfinite(MODELOS[_k])
_mod = MODELOS[_k] * (np.nansum(IMGS[_k][_s]) / np.nansum(MODELOS[_k][_s]))
_mask = source_mask((NY, NX), [(CY, CX)], 9.0)
_smooth = float((QC_C1.get('hybrid') or {}).get('smoothing_scale_px', 12.0))
_rad_h, _perf_h = radial_hybrid_profile(IMGS[_k] - _mod, (SY, SX), mask=_mask,
                                        smoothing_scale_px=_smooth)
_hyb = evaluate_radial_profile((NY, NX), (SY, SX), _rad_h, _perf_h)
VARIANTES = {'base (lo entregado)': _mod,
             'híbrido completo': _mod + _hyb,
             f'híbrido r > {R_NORM:.0f} px': _mod + np.where(RR > R_NORM, _hyb, 0.0)}
fig, axes = plt.subplots(1, 3, figsize=(12.5, 4.2))
_esc = np.nanpercentile(np.abs((IMGS[_k] - _mod)[RR > R_NORM]), 98)
for ax, (nom, m) in zip(axes, VARIANTES.items()):
    ax.imshow(IMGS[_k] - m, origin='lower', cmap='RdBu_r', vmin=-_esc, vmax=_esc)
    ax.add_patch(plt.Circle((SX, SY), R_NORM, fill=False, color='k', lw=.9, ls='--'))
    ax.plot(CX, CY, 'x', color='lime', ms=9, mew=2)
    ax.set_title(nom, fontsize=9); ax.set_xticks([]); ax.set_yticks([])
fig.suptitle(f'residuo a {LAMS[_k]:.0f} Å con cada variante')
plt.tight_layout(); plt.show()

print(f'{"variante":26s} {"anillo %":>9s} {"núcleo/total %":>15s}')
for nom, m in VARIANTES.items():
    _a = companion_ring_metric(IMGS[_k], m, (SY, SX), (CY, CX), width_px=3.0,
                               source_exclusion_radius_px=9.0)['median_pct']
    _e = encircled_energy_metric(IMGS[_k], m, (SY, SX), norm_radius_px=R_NORM,
                                 box_half=1, exclude_mask=_mask)
    print(f'{nom:26s} {_a:9.2f} {_e.get("core_ratio_error_pct", float("nan")):15.2f}')
print('\nla tercera fila tiene el anillo de la segunda y el núcleo de la primera:')
print('toda la mejora viene de FUERA del radio de normalización, y todo el daño')
print('a la corrección de apertura venía de DENTRO.')


### 8.a · Pero la mitad de esa mejora es circular

El híbrido se construye como `dato − modelo` y se suma al modelo. Medir después el residuo **sobre el mismo dato** tiene la bajada garantizada por construcción: es un ajuste, no una predicción.

La comprobación honesta es **partir el dato**: ajustar el híbrido en una mitad de las exposiciones y evaluarlo en la otra. Aquí se hace con las dos mitades del cubo por bandas alternas de λ, que es la versión barata del mismo argumento; la versión con exposiciones disjuntas dio que **generaliza en torno a la mitad**.

> Lo que **no** es circular es el núcleo: que la corrección de apertura no se mueva sale de que $F(\le R_{\rm norm})$ no se toca. Eso es geometría.


In [ ]:
_A = [k for k in range(len(LAMS)) if k % 2 == 0]
_B = [k for k in range(len(LAMS)) if k % 2 == 1]
def _mod_norm(k):
    _s = (RR <= R_NORM) & np.isfinite(IMGS[k]) & np.isfinite(MODELOS[k])
    return MODELOS[k] * (np.nansum(IMGS[k][_s]) / np.nansum(MODELOS[k][_s]))
def _perfil_de(indices):
    _acc = []
    for k in indices:
        _r, _p = radial_hybrid_profile(IMGS[k] - _mod_norm(k), (SY, SX),
                                       mask=_mask, smoothing_scale_px=_smooth)
        _acc.append(_p)
    return _r, np.nanmedian(np.asarray(_acc), axis=0)
_rA, _pA = _perfil_de(_A)
_kB = _B[len(_B) // 2]
_mB = _mod_norm(_kB)
_rB, _pB = _perfil_de([_kB])
def _anillo_con(perf, radios):
    _h = evaluate_radial_profile((NY, NX), (SY, SX), radios, perf)
    return companion_ring_metric(IMGS[_kB], _mB + np.where(RR > R_NORM, _h, 0.0),
                                 (SY, SX), (CY, CX), width_px=3.0,
                                 source_exclusion_radius_px=9.0)['median_pct']
_base = companion_ring_metric(IMGS[_kB], _mB, (SY, SX), (CY, CX), width_px=3.0,
                              source_exclusion_radius_px=9.0)['median_pct']
_dentro = _anillo_con(_pB, _rB)
_fuera = _anillo_con(_pA, _rA)
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.bar(['base', 'ajustado en\nel mismo dato\n(circular)',
        'ajustado en\notras bandas\n(honesto)'],
       [_base, _dentro, _fuera],
       color=['0.6', '#d62728', '#2ca02c'])
ax.set_ylabel('residuo de anillo [%]')
ax.set_title(f'evaluado siempre en la banda de {LAMS[_kB]:.0f} Å')
ax.grid(alpha=.3, axis='y'); plt.tight_layout(); plt.show()
_gen = (_base - _fuera) / (_base - _dentro) if (_base - _dentro) else float('nan')
print(f'base {_base:.2f} %  ·  circular {_dentro:.2f} %  ·  honesto {_fuera:.2f} %')
print(f'generaliza el {100 * _gen:.0f} % de la mejora')


## 9 · A quién le importa este error, y a quién no

Aquí se cierra el círculo con la §3. El error del halo vive **fuera** del radio de normalización, así que:

- **no toca la fotometría de apertura**, porque ese tramo lo mide A2 empíricamente;
- **sí muerde** a los métodos que usan el modelo como **fondo espacial**.

El síntoma con el que empezó todo es exactamente eso: con señal inyectada **nula**, un método devuelve flujo positivo en casi todas las posiciones de control — un pedestal — y el otro no.

> **Cuidado con este párrafo: el pedestal resultó NO ser este error.** Lo que sigue midiendo esta sección es real —el pedestal existe y se mide aquí— pero atribuirlo al déficit del halo es lo que la §10 falsa, con la medida del 2026-08-29. Se conserva porque es el hilo por el que se llegó, no porque la atribución fuera correcta.


In [ ]:
if not CSV_PEDESTAL.exists():
    print('sin', CSV_PEDESTAL.name, ': esta sección necesita la tabla precalculada.')
else:
    _p = [x for x in csv.DictReader(open(CSV_PEDESTAL))
          if x['position_label'] != 'real']
    _mets = sorted({x['method'] for x in _p})
    _noches = sorted({x['noche'] for x in _p})
    # El estadístico es la SNR, no el flujo: el flujo lleva dentro la
    # `apcorr`, que difiere mucho entre exposiciones, así que los flujos de
    # dos exposiciones NO son comparables. La SNR es adimensional, y además
    # es lo que define un falso positivo.
    fig, (c1, c2) = plt.subplots(1, 2, figsize=(11.5, 3.8))
    for _m in _mets:
        _s = [float(x['recovered_snr']) for x in _p if x['method'] == _m]
        c1.hist(_s, bins=40, range=(-8, 12), histtype='step', lw=2, label=_m)
    c1.axvline(5, color='crimson', ls='--', label='umbral de detección')
    c1.set_xlabel('SNR recuperada con señal NULA'); c1.set_ylabel('medidas')
    c1.set_title('lo que cae a la derecha del umbral\nes un falso positivo')
    c1.legend(fontsize=8); c1.grid(alpha=.3)
    _x = np.arange(len(_mets)); _w = .8 / max(len(_noches), 1)
    for _i, _n in enumerate(_noches):
        _tas = []
        for _m in _mets:
            _f = [x for x in _p if x['method'] == _m and x['noche'] == _n]
            _tas.append(100 * sum(1 for x in _f
                                  if float(x['recovered_snr']) >= 5) / len(_f))
        c2.bar(_x + _i * _w, _tas, _w, label=_n)
    c2.set_xticks(_x + _w * (len(_noches) - 1) / 2); c2.set_xticklabels(_mets)
    c2.set_ylabel('falsos positivos [%]')
    c2.set_title('no es del sustrato: es del estimador')
    c2.legend(fontsize=8); c2.grid(alpha=.3, axis='y')
    plt.tight_layout(); plt.show()

    print(f"{'método':10s} {'falsos positivos':>20s} {'tasa':>7s}"
          f" {'SNR mediana':>12s}")
    for _m in _mets:
        _f = [x for x in _p if x['method'] == _m]
        _s = [float(x['recovered_snr']) for x in _f]
        _fp = sum(1 for q in _s if q >= 5)
        print(f'{_m:10s} {_fp:9d} / {len(_f):<8d}'
              f' {100 * _fp / len(_f):6.1f}% {np.median(_s):12.2f}')
    print('\nel mismo dato, las mismas posiciones, la misma señal nula:'
          ' la diferencia es el estimador.')


> **Una comparación que no cuadra con la intuición.** Sobre el cubo **combinado** el mismo método da una tasa de falsos positivos **varias veces mayor** que la de aquí. O sea: el déficit del modelo es peor **por exposición** (§5), pero el pedestal que produce es peor **en el combinado**. Son dos cosas distintas y van en direcciones opuestas — conviene no confundirlas.

> **Y una advertencia de método, pagada con una medida perdida.** Corregir el halo *dentro del modelo de PSF* no funciona: el término es aditivo y no cae con el radio, así que aplicado a la plantilla de una fuente puntual la vuelve casi plana. La corrección es una propiedad de la **escena** —el halo residual de la primaria, en su posición— y no de la **función PSF**, que se evalúa también para el objeto, para la inyección y para la corrección de apertura. C1 lo hace bien: suma a la **imagen** del modelo por bin, no al evaluador.


## 10 · El pedestal no era esto (medido el 2026-08-29)

Todo lo anterior mide un déficit **real** del modelo en el halo. Lo que **no** es cierto es la atribución con la que se cerraba la §9: que ese déficit fuera la causa del pedestal. `docs/2026-08-29_frente4_pedestal_y_overfit.md`, sonda reproducible en `scripts/psffit_pedestal_probe.py`.

**El anillo no predice el pedestal, y va al revés.** ROXs 42B b tiene el anillo del modelo entregado **9× peor** que ROXs 12 b (61.5 % contra 6.7 %) y **no tiene pedestal** (+0.40 σ contra +4.86 σ). La celda de abajo comprueba las dos cantidades para *este* objeto, en vivo.

**Lo que sí lo causa es la rigidez del fondo del ajuste.** `psf_pair_design` lleva constante y **dos gradientes lineales**; lo que el modelo deja en el anillo tiene curvatura a la escala del disco de ajuste, y lo que los lineales no absorben se lo lleva la amplitud del compañero. Con términos cuadráticos el pedestal de ROXs 12 b pasa de +91.09 a +10.66 — de **+4.86 σ a +0.29 σ**: el **88 %** del pedestal es el fondo.

**Y no se puede quitar así, porque se lleva la señal.** Medido con el exceso del compañero sobre su propia nula partido por el ruido entre posiciones, los diseños alternativos **empeoran en los dos objetos**: en ROXs 12 b la SNR del compañero pasa de **+3.14** a +1.04 (fondo cuadrático) o +1.70 (término de halo de 1 gdl). El cuadrático se lleva el 88 % del pedestal **y el 68 % de la señal**.

Es **estructural**: el compañero y los controles están **al mismo radio**, y sobre un disco de 12 px una fuente puntual y un fondo curvo son casi degenerados. El pedestal **no es un sesgo del estimador: es el nivel de referencia local**, y restarlo tiene un sitio correcto que no es el ajuste — la **estandarización de E4 v4**, que resta la mediana de las nulas medida al mismo radio, **después** de extraer y sin coste en señal.


In [ ]:
# Las dos cantidades que la §10 enfrenta, para ESTE objeto y en vivo.
_e01 = SD / 'stage_e01_qc.json'
_h04 = SD / 'stage_h04_qc.json'
if not (_e01.exists() and _h04.exists()):
    print('faltan', _e01.name, 'o', _h04.name, ': sección informativa.')
else:
    _q = json.loads(_e01.read_text())
    _mix = (((_q.get('per_observation') or {}).get('delivered_vs_combined_fit') or {})
            .get('mixture_ring_residual_pct_median'))
    _ao = ((_q.get('model_comparison') or {}).get('psfao') or {}).get('ring_residual_pct_median')
    _nd = ((json.loads(_h04.read_text()).get('snr_standardization') or {})
           .get('null_distribution') or {})
    _ped = ((_nd.get('psffit') or {}).get('raw') or {})
    print(f"anillo del modelo entregado (mezcla): {_mix if _mix is None else round(_mix, 2)} %")
    print(f"anillo de la forma psfao          : {_ao if _ao is None else round(_ao, 2)} %")
    _m, _s = _ped.get('mean'), _ped.get('sigma')
    print(f"pedestal en la SNR CRUDA de E4    : media {None if _m is None else round(_m, 2)}"
          f" sigma {None if _s is None else round(_s, 2)}"
          f"  (n={(_nd.get('psffit') or {}).get('n')} controles)")
    print()
    print('El contraste entre objetos, con el OTRO estadístico: el cociente entre el')
    print('pedestal y la dispersión entre posiciones del ajuste a Halfa, que mide la')
    print('sonda scripts/psffit_pedestal_probe.py. No es el de arriba, aunque los dos')
    print('midan el pedestal y coincidan de cerca (docs/2026-08-29_frente4_...md §1):')
    print('   ROXs 12 b : anillo  6.7 %   pedestal/dispersion +4.86')
    print('   ROXs 42B b: anillo 61.5 %   pedestal/dispersion +0.40')
    print('   -> el del anillo 9x peor es el que NO tiene pedestal.')


## 11 · Qué NO decide este notebook

- **No aplica el híbrido restringido.** Lo mide y deja las dos caras: a favor, que la corrección de apertura no se mueve; en contra, que solo generaliza la mitad. Aplicarlo es un cambio en C1 y arrastra toda la cadena C→G.
- **No dice que el déficit desaparezca con otra forma de PSF.** Mide el modelo que la cadena entrega hoy, con la convención que la cadena usa.
- **No decide qué hacer con el método afectado.** Publica el pedestal medido; la decisión de si su número es utilizable es de quien firme el resultado.

> Lo que sí queda cerrado, con medida y no con argumento: el déficit del halo **no** lo crea la combinación, **no** es el radio de normalización, **no** es la ponderación y **no** es el cielo — y **no** es la causa del pedestal (§10).

> **Lo que sigue abierto es el halo mismo**, no el pedestal: el modelo reproduce sólo ~28 % del cromatismo medido y falla en el azul. Eso no lo arregla nada de lo probado aquí, y sigue mordiendo a los métodos que usan el modelo como fondo espacial.
